# Stage 2 Notebook 37 - Exp2FF KD self-distillation + cosine LR (NB35 retry, two bugs fixed)

**Why this exists.** NB35 (Exp2DD) failed at startup:

```
TypeError: LaneQueryHead.__init__() got an unexpected keyword argument 'num_priors'
```

`load_lane_teacher` in `train_joint_model_experiment.py` was hardcoded to `CurveLaneHead`'s constructor signature (`num_priors=...`) which `LaneQueryHead` doesn't accept. **Fixed**: the function now uses `inspect.signature(head_cls.__init__)` to filter only kwargs the head accepts. Works for any head type.

Combined with NB34's diagnosis (constant-LR divergence at epoch 22+), Exp2FF adds the cosine LR schedule from Exp2EE. So this experiment finally runs the project's namesake KD with both fixes in place.

Diff vs Exp2DD (broken NB35):
- `load_lane_teacher` is fixed (no config change, just code fix).
- New `train.lr_scheduler: cosine` (Exp2EE's fix).
- `train.end_epoch: 15 -> 20` (cosine decay window).
- `loss.lane.w_distill: 1.0` (KD active).
- `teacher.lane_head_checkpoint: <Exp2Z best.pt path>` (same teacher as NB35).

**Setup**: NB35's teacher-tar extraction cell already ran successfully on this Drive (the `.pt` file exists), so the cell below is a no-op pass-through. Run it anyway to confirm the path before training.

Reference: KD = CLRKD = the project's literal name. This is the first run that actually exercises it.

### Run mode

1. Confirm the teacher .pt path exists (cell below).
2. Keep `DEBUG_MODE = True` for the first run.
3. After smoke + debug pass, change to `False` for the 20-epoch short run.
4. Output mirrored to notebook cell, Colab runtime log, Drive log file.
5. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

# Confirm teacher checkpoint exists (it should - NB35 already extracted it).
import tarfile, shutil
TEACHER_TAR = '/content/drive/MyDrive/EcoCAR/training_runs/exp26_rmt_gca_mask_uncertainty_weighting_joint_short15.tar'
TEACHER_DEST = '/content/drive/MyDrive/EcoCAR/training_runs/exp26_rmt_gca_mask_uncertainty_weighting_joint_short15_best.pt'

if os.path.exists(TEACHER_DEST):
    print(f'Teacher checkpoint already at: {TEACHER_DEST}', flush=True)
else:
    if not os.path.exists(TEACHER_TAR):
        raise FileNotFoundError(f'Teacher tar not found: {TEACHER_TAR}. Run NB31 (Exp2Z) first.')
    print(f'Extracting best.pt from teacher tar: {TEACHER_TAR}', flush=True)
    with tarfile.open(TEACHER_TAR, 'r') as tar:
        for member in tar.getmembers():
            if member.name.endswith('best.pt'):
                with tar.extractfile(member) as src, open(TEACHER_DEST, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
                print(f'Extracted to: {TEACHER_DEST}', flush=True)
                break
        else:
            raise FileNotFoundError('best.pt not found in teacher tar.')

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane
Teacher checkpoint already at: /content/drive/MyDrive/EcoCAR/training_runs/exp26_rmt_gca_mask_uncertainty_weighting_joint_short15_best.pt


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp32_rmt_gca_mask_self_distillation_cosine_lr_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp32_rmt_gca_mask_self_distillation_cosine_lr_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp32_rmt_gca_mask_self_distillation_cosine_lr_joint_smoke.log
OK exp32_rmt_gca_mask_self_distillation_cosine_lr_joint.yaml
  lane_shape=(1, 12, 72, 2) det_shape=(1, 4, 4)
  lane_loss=3.2769 det_loss=3.0791 grad_cos=-0.1037 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.4996660649776459, 'gate/lane_mean': 0.49960434436798096, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp32_rmt_gca_mask_self_distillation_cosine_lr_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short20'
    EPOCHS = 20
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp32_rmt_gca_mask_self_distillation_cosine_lr_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp32_rmt_gca_mask_self_distillation_cosine_lr_joint_short20 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp32_rmt_gca_mask_self_distillation_cosine_lr_joint_short20.tar --epochs 20 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp32_rmt_gca_mask_self_distillation_cosine_lr_joint_short20.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp32_rmt_gca_mask_self_distillation_cosine_lr_joint_short20_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp32_rmt_gca_mask_s

0

## What to watch in Exp2FF training

Pass criteria at epoch 20:
- **`loaded lane teacher=...`** log line appears at the start of training (NOT a stack trace). Confirms `load_lane_teacher` fix works for `LaneQueryHead`.
- **`val/lane/distill`** loss component is logged and decreases over training (KD signal is being used).
- **`val/lane/decoded_f1 >= 0.07`** -- KD lifts cls toward oracle ceiling. Beats every previous Exp2 by ~60%.
- **`val/lane_exist_best_f1 >= 0.70`** -- teacher's calibrated cls signal improves student ranking.
- **Geometry holds**: `val/matched_line_iou >= 0.16`.
- **No late-epoch collapse** (cosine LR fix from Exp2EE).

Failure signals:
- `decoded_f1` plateaus at Exp2Z's 0.044 even with KD: teacher's signal isn't better than the loss function's. We've reached the architecture's true capacity. Pivot to bigger backbone or higher resolution.
- KD loss explodes: w_distill=1.0 too high; reduce to 0.5 or 0.25.
- Teacher loading still errors: check `load_lane_teacher`'s fix and which head class is being used.